In [1]:
import pandas as pd
import numpy as np
import plotly.express as px
from plotly.subplots import make_subplots
import ipywidgets as widgets
from ipywidgets import interact

In [2]:
bank = pd.read_csv("bank-full.csv", sep=";")
print(bank.columns)
print(bank.info())

Index(['age', 'job', 'marital', 'education', 'default', 'balance', 'housing',
       'loan', 'contact', 'day', 'month', 'duration', 'campaign', 'pdays',
       'previous', 'poutcome', 'y'],
      dtype='str')
<class 'pandas.DataFrame'>
RangeIndex: 45211 entries, 0 to 45210
Data columns (total 17 columns):
 #   Column     Non-Null Count  Dtype
---  ------     --------------  -----
 0   age        45211 non-null  int64
 1   job        45211 non-null  str  
 2   marital    45211 non-null  str  
 3   education  45211 non-null  str  
 4   default    45211 non-null  str  
 5   balance    45211 non-null  int64
 6   housing    45211 non-null  str  
 7   loan       45211 non-null  str  
 8   contact    45211 non-null  str  
 9   day        45211 non-null  int64
 10  month      45211 non-null  str  
 11  duration   45211 non-null  int64
 12  campaign   45211 non-null  int64
 13  pdays      45211 non-null  int64
 14  previous   45211 non-null  int64
 15  poutcome   45211 non-null  str  
 16  y   

In [3]:
px.histogram(bank, x="duration") # y - liczba obserwacji

In [4]:
bank_positive_y = bank[(bank["y"] == "yes")]
bank_negative_y = bank[(bank["y"] == "no")]
#print(bank_positive_y)

positive_y_plot = px.histogram(bank_positive_y, x="duration")
negative_y_plot = px.histogram(bank_negative_y, x="duration")
fig = make_subplots(
    rows=1,
    cols=2,
    subplot_titles=("Klienci, którzy wykupili lokatę", 
                    "Klienci, którzy nie wykupili lokaty")
)

for trace in positive_y_plot.data:
    fig.add_trace(trace, row=1, col=1)

for trace in negative_y_plot.data:
    fig.add_trace(trace, row=1, col=2)

fig.update_xaxes(title_text="Czas trwania rozmowy", row=1, col=1)
fig.update_xaxes(title_text="Czas trwania rozmowy", row=1, col=2)

fig.update_yaxes(title_text="Liczba rozmów", row=1, col=1)
fig.update_yaxes(title_text="Liczba rozmów", row=1, col=2)

fig.show()

Klienci, którzy końcowo nie aktywowali lokaty mieli mniejszy czas rozmowy, niz ci co byli zainteresowani propozycją.
Ogólnie duzo było klientów, rozmowa z któtymi wynosiła pmiędzy 50 a 190 sekund.

In [5]:
bank_above_25 = bank[bank["age"] > 25]
#print(bank_above_25)
px.histogram(bank_above_25, x="balance")

In [6]:
def show_histogram(age):
    bank_above_age = bank[bank["age"] > age]

    fig = px.histogram(
        bank_above_age,
        x="balance",
        title=f"Rozkład salda konta osób powyżej {age} lat",
        labels={
            "balance": "Saldo konta",
            "count": "Liczba klientów"
        }
    )

    fig.show()

interact(
    show_histogram,
    age=widgets.IntSlider(
        value=25,
        min=18,
        max=int(bank["age"].max()),
        step=1,
        description="Wiek:",
        continuous_update=False
    )
)

interactive(children=(IntSlider(value=25, continuous_update=False, description='Wiek:', max=95, min=18), Outpu…

<function __main__.show_histogram(age)>

In [7]:
t_threshold = 360
bank_duration_threshold = bank[bank["duration"] > t_threshold]
bank_duration_threshold_n = len(bank_duration_threshold)

bank_duration_threshold_positive = bank[(bank["duration"] > t_threshold) & (bank["y"] == "yes")]
bank_duration_threshold_positive_n = len(bank_duration_threshold_positive)

print("Procent pozytywnych próbek przy duration = 360: ", 
      bank_duration_threshold_positive_n / bank_duration_threshold_n * 100)

Procent pozytywnych próbek przy duration = 360:  32.56360915116528


In [8]:
duration_max = bank["duration"].max()

results = []

for t in range(0, duration_max, 15):
    filtered = bank[bank["duration"] > t]

    if len(filtered) > 0:
        percentage = (filtered["y"] == "yes").mean() * 100

        results.append({
            "t": t,
            "percentage": percentage
        })

results_df = pd.DataFrame(results)

fig = px.scatter(
    results_df,
    x="t",
    y="percentage",
    title="Procent pozytywnych próbek w zależności od progu czasu rozmowy"
)

fig.show()

In [9]:
balance_max = bank["balance"].max()

results = []

for t in range(0, balance_max, 500):
    filtered = bank[bank["balance"] > t]

    if len(filtered) > 0:
        percentage = (filtered["y"] == "yes").mean() * 100

        results.append({
            "t": t,
            "percentage": percentage
        })

results_df = pd.DataFrame(results)

fig = px.scatter(
    results_df,
    x="t",
    y="percentage",
    title="Procent pozytywnych próbek w zależności od salda"
)

fig.show()

In [10]:
p_all = (bank["y"] == "yes").mean()

bank_over_60k = bank[bank["balance"] > 60000]
p_over_60k = (bank_over_60k["y"] == "yes").mean()

bank_between_60k_82k = bank[
    (bank["balance"] > 60000) &
    (bank["balance"] < 82000)
]
p_between_60k_82k = (bank_between_60k_82k["y"] == "yes").mean()

print(f"Cały zbiór: {p_all}")
print(f"Balance > 60k: {p_over_60k}")
print(f"60k < balance < 82k: {p_between_60k_82k}")

Cały zbiór: 0.11698480458295547
Balance > 60k: 0.25
60k < balance < 82k: 0.3333333333333333


Prawdopodobieństwo, ze losowa osoba ze zbioru będzie chciała lokatę wynosi ~11%,
a osoba z saldem powyzej 60k: 25%. Jeśli mamy wybrać z tych dwóch strategii, 2 jest lepsza.
Natomiast, gdy saldo klienta jest w ramach (60k, 82k) prawdopobieństwo zainteresowania propozycją wynosi ~33%,
co jest jeszcze lepszym wynikiem

In [11]:
first_half = bank.sample(frac=0.5)
second_half = bank.drop(first_half.index)

In [12]:
balance_max = first_half["balance"].max()
results = []

for t in range(0, balance_max, 500):
    filtered = bank[bank["balance"] > t]

    if len(filtered) > 0:
        percentage = (filtered["y"] == "yes").mean() * 100

        results.append({
            "t": t,
            "percentage": percentage
        })

results_df = pd.DataFrame(results)

fig = px.scatter(
    results_df,
    x="t",
    y="percentage",
    title="Procent pozytywnych próbek w zależności od salda"
)

fig.show()

In [13]:
def calc_errors(data, t):
    predicted_positive = data["balance"] > t
    predicted_negative = data["balance"] <= t

    actual_positive = data["y"] == "yes"
    actual_negative = data["y"] == "no"

    false_positive = predicted_positive & actual_negative
    false_negative = predicted_negative & actual_positive

    predicted_positive_n = predicted_positive.sum()
    actual_positive_n = actual_positive.sum()

    error_1 = (
        false_positive.sum() / predicted_positive_n * 100
        if predicted_positive_n > 0 else 0
    )

    error_2 = (
        false_negative.sum() / actual_positive_n * 100
        if actual_positive_n > 0 else 0
    )

    return error_1, error_2

In [14]:
my_t = 3100
error_1, error_2 = calc_errors(first_half, my_t)
print("Zbiór treningowy: ")
print("Ilu klientów nie było jednak zainteresowanych", error_1, "Ilu zainteresowanych klientów pominęliśmy: ", error_2)

error_1, error_2 = calc_errors(second_half, my_t)
print("Zbiór testowy: ")
print("Ilu klientów nie było jednak zainteresowanych", error_1, "Ilu zainteresowanych klientów pominęliśmy: ", error_2)

Zbiór treningowy: 
Ilu klientów nie było jednak zainteresowanych 83.58047016274864 Ilu zainteresowanych klientów pominęliśmy:  82.64525993883792
Zbiór testowy: 
Ilu klientów nie było jednak zainteresowanych 83.28290468986384 Ilu zainteresowanych klientów pominęliśmy:  83.46427235316123


In [15]:
thresholds = np.arange(
    first_half["balance"].min(),
    first_half["balance"].max() + 100,
    100
)

results_first_half = []

for t in thresholds:
    error_1, error_2 = calc_errors(first_half, t)

    results_first_half.append({
        "threshold": t,
        "false_positive_pct": error_1,
        "false_negative_pct": error_2
    })

first_half_results = pd.DataFrame(results_first_half)

In [16]:
px.line(
    first_half_results,
    x="threshold",
    y=["false_positive_pct", "false_negative_pct"],
    markers=True,
    title="Błędy modelu w zależności od progu salda, zbiór treningowy"
)

In [17]:
results_second_half = []

for t in thresholds:
    error_1, error_2 = calc_errors(second_half, t)

    results_second_half.append({
        "threshold": t,
        "false_positive_pct": error_1,
        "false_negative_pct": error_2
    })

first_half_results = pd.DataFrame(results_second_half)

In [18]:
px.line(
    results_second_half,
    x="threshold",
    y=["false_positive_pct", "false_negative_pct"],
    markers=True,
    title="Błędy modelu w zależności od progu salda, zbiór testowy"
)

Uwazam, ze saldo nie powinno być jednym warunkiem przez który estymujemy, czy klient będzie zainteresowany, 
ale saldo powyzej 3100 wydaje się być sensowne. Jest to tez miejsce, gdzie 2 wykresy się przecinają.